In [1]:
%load_ext autoreload
%autoreload 2

import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

from src.data.db import get_connection
from src.processing.xg_processing import build_xg_features, TARGET_COL

feat = build_xg_features(get_connection())
print(len(feat))   # should still be about 562,000
print(feat[["dist_from_prev", "speed_from_prev", "rebound_angle_change"]].describe().round(1))

562374
       dist_from_prev  speed_from_prev  rebound_angle_change
count        559590.0         559590.0               53139.0
mean             60.0              8.1                  43.0
std              45.6             10.1                  33.7
min               0.0              0.0                   0.0
25%              25.2              2.1                  15.3
50%              45.0              4.8                  35.6
75%              84.5             10.4                  64.7
max             198.1            193.7                 180.0


In [2]:
TEST_SEASON = 20252026
train = feat[feat.season < TEST_SEASON]
test  = feat[feat.season == TEST_SEASON]
print(f"train: {len(train):,}   test: {len(test):,}")

train: 451,275   test: 111,099


In [3]:
BASE_COLS = ["distance", "angle"]

base = LogisticRegression(max_iter=1000)
base.fit(train[BASE_COLS], train[TARGET_COL])

test_pred = base.predict_proba(test[BASE_COLS])[:, 1]

In [4]:
naive = np.full(len(test), train[TARGET_COL].mean())   # every shot = league average

print(f"AUC:       {roc_auc_score(test[TARGET_COL], test_pred):.3f}")
print(f"log loss:  {log_loss(test[TARGET_COL], test_pred):.4f}")
print(f"  naive:   {log_loss(test[TARGET_COL], naive):.4f}")
print(f"predicted goals: {test_pred.sum():,.0f}   actual: {test[TARGET_COL].sum():,}")
print("weights:", dict(zip(BASE_COLS, base.coef_[0].round(4))))

AUC:       0.714
log loss:  0.2307
  naive:   0.2490
predicted goals: 7,687   actual: 7,578
weights: {'distance': np.float64(-0.0473), 'angle': np.float64(-0.0127)}


In [5]:
cal = pd.DataFrame({"xg": test_pred, "goal": test[TARGET_COL].values})
cal["bin"] = pd.qcut(cal.xg, 10, labels=False)   # 10 equal-size groups by xG

print(cal.groupby("bin").agg(
    shots=("goal", "size"),
    predicted=("xg", "mean"),
    actual=("goal", "mean"),
).round(3))

     shots  predicted  actual
bin                          
0    11123      0.009   0.006
1    11105      0.019   0.016
2    11108      0.027   0.020
3    11111      0.036   0.035
4    11124      0.046   0.054
5    11102      0.058   0.073
6    11161      0.076   0.096
7    11058      0.102   0.106
8    11435      0.135   0.130
9    10772      0.185   0.147


In [6]:
import lightgbm as lgb
from src.processing.xg_processing import FEATURE_COLS

VALID_SEASON = 20242025
gbm_train = feat[feat.season < VALID_SEASON]
gbm_valid = feat[feat.season == VALID_SEASON]
# test is the same 2025-26 set the baseline used

print(f"train: {len(gbm_train):,}   valid: {len(gbm_valid):,}   test: {len(test):,}")

train: 339,971   valid: 111,304   test: 111,099


In [7]:
gbm = lgb.LGBMClassifier(
    n_estimators=2000,        # upper limit on trees; early stopping picks the real number
    learning_rate=0.03,       # how much each tree corrects; smaller = slower but steadier
    num_leaves=31,            # how detailed each tree can get
    min_child_samples=200,    # every pattern must be backed by at least 200 shots
    subsample=0.8, subsample_freq=1,   # each tree sees a random 80% of shots
    colsample_bytree=0.8,              # ...and a random 80% of features
    verbose=-1,
)
gbm.fit(
    gbm_train[FEATURE_COLS], gbm_train[TARGET_COL],
    eval_set=[(gbm_valid[FEATURE_COLS], gbm_valid[TARGET_COL])],
    eval_metric="binary_logloss",
    callbacks=[lgb.early_stopping(100, verbose=False)],
)
print("trees used:", gbm.best_iteration_)

trees used: 412


In [8]:
# 1. Validation run: weight the training seasons, compare with the unweighted model
w_train = gbm_train.season.map({20212022: 1.0, 20222023: 1.25, 20232024: 1.5})

gbm_w = lgb.LGBMClassifier(**gbm.get_params())
gbm_w.fit(
    gbm_train[FEATURE_COLS], gbm_train[TARGET_COL],
    sample_weight=w_train,
    eval_set=[(gbm_valid[FEATURE_COLS], gbm_valid[TARGET_COL])],
    eval_metric="binary_logloss",
    callbacks=[lgb.early_stopping(100, verbose=False)],
)

def redirect_ratio(df, pred):
    r = df.shot_type == "redirect"
    return pred[r].sum() / df.loc[r, TARGET_COL].sum()

for name, m in [("unweighted", gbm), ("weighted", gbm_w)]:
    vp = m.predict_proba(gbm_valid[FEATURE_COLS])[:, 1]
    print(f"{name:11} valid log loss {log_loss(gbm_valid[TARGET_COL], vp):.4f}   "
          f"pred/actual {vp.sum() / gbm_valid[TARGET_COL].sum():.3f}   "
          f"redirect ratio {redirect_ratio(gbm_valid, vp):.2f}")

w_all = train.season.map({20212022: 1.0, 20222023: 1.25, 20232024: 1.5, 20242025: 2.0})
final_w = lgb.LGBMClassifier(**{**gbm_w.get_params(), "n_estimators": gbm_w.best_iteration_})
final_w.fit(train[FEATURE_COLS], train[TARGET_COL], sample_weight=w_all)

fp = final_w.predict_proba(test[FEATURE_COLS])[:, 1]
print(f"AUC {roc_auc_score(test[TARGET_COL], fp):.3f}   log loss {log_loss(test[TARGET_COL], fp):.4f}   "
      f"predicted {fp.sum():,.0f}   actual {test[TARGET_COL].sum():,}   "
      f"redirect ratio {redirect_ratio(test, fp):.2f}")

unweighted  valid log loss 0.2134   pred/actual 0.984   redirect ratio 1.13
weighted    valid log loss 0.2125   pred/actual 0.982   redirect ratio 1.12
AUC 0.769   log loss 0.2167   predicted 7,762   actual 7,578   redirect ratio 1.24


In [9]:
gbm_pred = gbm.predict_proba(test[FEATURE_COLS])[:, 1]

print(f"{'':12}{'baseline':>10}{'LightGBM':>10}")
print(f"{'AUC':12}{roc_auc_score(test[TARGET_COL], test_pred):>10.3f}"
      f"{roc_auc_score(test[TARGET_COL], gbm_pred):>10.3f}")
print(f"{'log loss':12}{log_loss(test[TARGET_COL], test_pred):>10.4f}"
      f"{log_loss(test[TARGET_COL], gbm_pred):>10.4f}")
print(f"predicted goals: {gbm_pred.sum():,.0f}   actual: {test[TARGET_COL].sum():,}")

              baseline  LightGBM
AUC              0.714     0.755
log loss        0.2307    0.2204
predicted goals: 7,777   actual: 7,578


In [10]:
cal = pd.DataFrame({"xg": gbm_pred, "goal": test[TARGET_COL].values})
cal["bin"] = pd.qcut(cal.xg, 10, labels=False)
print(cal.groupby("bin").agg(
    shots=("goal", "size"),
    predicted=("xg", "mean"),
    actual=("goal", "mean"),
).round(3))

     shots  predicted  actual
bin                          
0    11110      0.005   0.005
1    11110      0.012   0.010
2    11110      0.019   0.019
3    11110      0.029   0.029
4    11110      0.042   0.042
5    11109      0.057   0.060
6    11110      0.076   0.084
7    11110      0.100   0.103
8    11110      0.133   0.125
9    11110      0.228   0.204


In [11]:
imp = pd.Series(gbm.booster_.feature_importance("gain"), index=FEATURE_COLS)
print((imp / imp.sum()).sort_values(ascending=False).round(3))

distance                 0.545
angle                    0.120
time_since_last_event    0.065
shot_type                0.058
rebound_angle_change     0.050
dist_from_prev           0.045
speed_from_prev          0.044
last_event_type          0.026
total_skaters            0.022
skater_diff              0.020
is_rebound               0.002
is_rush                  0.001
own_goalie_pulled        0.001
dtype: float64


In [12]:
print(feat.groupby("season")[TARGET_COL].agg(shots="size", goal_rate="mean").round(4))

valid_pred = gbm.predict_proba(gbm_valid[FEATURE_COLS])[:, 1]
print(f"\n2024-25 predicted: {valid_pred.sum():,.0f}   actual: {gbm_valid[TARGET_COL].sum():,}")

           shots  goal_rate
season                     
20212022  112561     0.0682
20222023  113351     0.0689
20232024  114059     0.0670
20242025  111304     0.0663
20252026  111099     0.0682

2024-25 predicted: 7,259   actual: 7,377


In [13]:
final = lgb.LGBMClassifier(**{**gbm.get_params(), "n_estimators": gbm.best_iteration_})
final.fit(train[FEATURE_COLS], train[TARGET_COL])      # train = all seasons before 2025-26

final_pred = final.predict_proba(test[FEATURE_COLS])[:, 1]
print(f"AUC {roc_auc_score(test[TARGET_COL], final_pred):.3f}   "
      f"log loss {log_loss(test[TARGET_COL], final_pred):.4f}   "
      f"predicted {final_pred.sum():,.0f}   actual {test[TARGET_COL].sum():,}")

AUC 0.765   log loss 0.2178   predicted 7,750   actual 7,578


In [14]:
recent = feat.season == 20252026

# 1. Do shot types look different in 2025-26?
st = pd.crosstab(feat.shot_type, recent, normalize="columns").round(3)
st.columns = ["2021-25", "2025-26"]
print(st, "\n")

# 2. Are gaps between events shorter in 2025-26?
print(feat.groupby(recent).time_since_last_event.describe().round(1), "\n")

# 3. Did last_event_type change?
le = pd.crosstab(feat.last_event_type, recent, normalize="columns").round(3)
le.columns = ["2021-25", "2025-26"]
print(le, "\n")

# 4. Where in 2025-26 does the over-prediction come from?
t = test.assign(xg=final_pred)
print(t.groupby("shot_type", observed=True)
       .agg(shots=("is_goal", "size"), predicted=("xg", "sum"), actual=("is_goal", "sum"))
       .assign(ratio=lambda d: (d.predicted / d.actual).round(2))
       .sort_values("shots", ascending=False))

              2021-25  2025-26
shot_type                     
backhand        0.072    0.072
bat             0.002    0.005
between-legs    0.000    0.001
cradle          0.000    0.000
poke            0.003    0.003
redirect        0.104    0.112
slap            0.124    0.119
wrap-around     0.008    0.007
wrist-snap      0.687    0.681 

           count  mean   std  min  25%   50%   75%    max
season                                                   
False   339090.0  14.6  14.6  0.0  4.0  10.0  20.0  234.0
True     84371.0  14.9  15.0  0.0  4.0  10.0  21.0  188.0 

                 2021-25  2025-26
last_event_type                  
blocked-shot       0.165    0.166
delayed-penalty    0.006    0.007
giveaway           0.125    0.175
hit                0.233    0.202
missed-shot        0.142    0.161
shot-on-goal       0.249    0.234
takeaway           0.081    0.054 

              shots    predicted  actual  ratio
shot_type                                      
wrist-snap    75623

In [15]:
r = feat[feat.shot_type == "redirect"]
print(r.groupby("season").agg(
    shots=("is_goal", "size"),
    goal_rate=("is_goal", "mean"),
    median_dist=("distance", "median"),
    share_under_10ft=("distance", lambda d: (d < 10).mean()),
).round(3))

          shots  goal_rate  median_dist  share_under_10ft
season                                                   
20212022  10732      0.094       13.342             0.248
20222023  11916      0.090       13.000             0.302
20232024  12421      0.083       12.530             0.320
20242025  11750      0.081       12.649             0.331
20252026  12493      0.075       12.000             0.382


In [16]:
print(gbm.best_score_["valid_0"]["binary_logloss"])   # 2024-25 validation log loss

0.2134320525465986


In [17]:
shot_teams = pd.read_sql("""
    SELECT se.game_id, se.event_id,
           CASE WHEN se.team_id = s.home_team_id
                THEN s.home_team_abbrev ELSE s.away_team_abbrev END AS team,
           CASE WHEN se.team_id = s.home_team_id
                THEN s.away_team_abbrev ELSE s.home_team_abbrev END AS opponent
    FROM shot_events se
    JOIN schedule s ON s.game_id = se.game_id
""", get_connection())

t = test.assign(xg=fp).merge(shot_teams, on=["game_id", "event_id"])
five = t[(t.skater_diff == 0) & (t.total_skaters == 10)]          # 5-on-5 only

ours = pd.DataFrame({
    "xGF": five.groupby("team").xg.sum(),
    "xGA": five.groupby("opponent").xg.sum(),
    "GF":  five.groupby("team").is_goal.sum(),
})
print(len(ours), "teams")          # expect 32

32 teams


In [19]:
mp_raw = pd.read_csv("../data/external/moneypuck_teams_2025.csv")
print(mp_raw.columns.tolist())
print(mp_raw.situation.unique())

['team', 'season', 'name', 'team.1', 'position', 'situation', 'games_played', 'xGoalsPercentage', 'corsiPercentage', 'fenwickPercentage', 'iceTime', 'xOnGoalFor', 'xGoalsFor', 'xReboundsFor', 'xFreezeFor', 'xPlayStoppedFor', 'xPlayContinuedInZoneFor', 'xPlayContinuedOutsideZoneFor', 'flurryAdjustedxGoalsFor', 'scoreVenueAdjustedxGoalsFor', 'flurryScoreVenueAdjustedxGoalsFor', 'shotsOnGoalFor', 'missedShotsFor', 'blockedShotAttemptsFor', 'shotAttemptsFor', 'goalsFor', 'reboundsFor', 'reboundGoalsFor', 'freezeFor', 'playStoppedFor', 'playContinuedInZoneFor', 'playContinuedOutsideZoneFor', 'savedShotsOnGoalFor', 'savedUnblockedShotAttemptsFor', 'penaltiesFor', 'penalityMinutesFor', 'faceOffsWonFor', 'hitsFor', 'takeawaysFor', 'giveawaysFor', 'lowDangerShotsFor', 'mediumDangerShotsFor', 'highDangerShotsFor', 'lowDangerxGoalsFor', 'mediumDangerxGoalsFor', 'highDangerxGoalsFor', 'lowDangerGoalsFor', 'mediumDangerGoalsFor', 'highDangerGoalsFor', 'scoreAdjustedShotsAttemptsFor', 'unblockedShot

In [21]:
# MoneyPuck writes a few abbreviations with dots
MP_TO_NHL = {"T.B": "TBL", "N.J": "NJD", "S.J": "SJS", "L.A": "LAK"}

mp = (mp_raw[mp_raw.situation == "5on5"]
      .assign(team=lambda d: d.team.replace(MP_TO_NHL))
      .set_index("team")[["xGoalsFor", "xGoalsAgainst"]])

both = ours.join(mp, how="inner")
print(len(both), "teams matched")                    # expect 32

print(f"xGF correlation: {both.xGF.corr(both.xGoalsFor):.3f}")
print(f"xGA correlation: {both.xGA.corr(both.xGoalsAgainst):.3f}")
print(f"total xG, ours / MoneyPuck: {both.xGF.sum() / both.xGoalsFor.sum():.3f}")

print(f"ours:      {both.xGF.sum():,.0f}")
print(f"MoneyPuck: {both.xGoalsFor.sum():,.0f}")
print(f"actual:    {both.GF.sum():,.0f}")

32 teams matched
xGF correlation: 0.970
xGA correlation: 0.940
total xG, ours / MoneyPuck: 1.000
ours:      5,376
MoneyPuck: 5,376
actual:    5,364


In [22]:
results = []
for leaves in [15, 31, 63]:
    for min_child in [100, 200, 500]:
        m = lgb.LGBMClassifier(**{**gbm_w.get_params(),
                                  "num_leaves": leaves,
                                  "min_child_samples": min_child})
        m.fit(gbm_train[FEATURE_COLS], gbm_train[TARGET_COL],
              sample_weight=w_train,
              eval_set=[(gbm_valid[FEATURE_COLS], gbm_valid[TARGET_COL])],
              eval_metric="binary_logloss",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        results.append({"num_leaves": leaves, "min_child": min_child,
                        "trees": m.best_iteration_,
                        "valid_logloss": m.best_score_["valid_0"]["binary_logloss"]})

print(pd.DataFrame(results).sort_values("valid_logloss").round(5).to_string(index=False))

 num_leaves  min_child  trees  valid_logloss
         15        100    834        0.21243
         31        500    304        0.21247
         15        500    592        0.21247
         31        100    414        0.21248
         31        200    387        0.21248
         15        200    575        0.21248
         63        100    271        0.21264
         63        200    271        0.21267
         63        500    277        0.21275
